In [1]:
import csv
import re
from collections import Counter
import math

In [3]:
#SKILL_ALIASES mapping
SKILL_ALIASES = {
    # Languages
    "python": "python",
    "pyhton": "python",
    "java": "java",
    "javascript": "javascript",
    "javascrpit": "javascript",
    "js": "javascript",
    "typescript": "typescript",
    "typescrpit": "typescript",
    "c++": "cpp",
    "cpp": "cpp",
    "r": "r",
    "kotlin": "kotlin",
    # ML / Data
    "machinelearning": "machine_learning",
    "machine learning": "machine_learning",
    "ml": "machine_learning",
    "sklearn": "machine_learning",
    "deeplearning": "deep_learning",
    "deep learning": "deep_learning",
    "deep-learning": "deep_learning",
    "tensorflow": "tensorflow",
    "pytorch": "pytorch",
    "keras": "keras",
    "nlp": "nlp",
    "bert": "bert",
    "xgboost": "xgboost",
    "feature engineering": "feature_engineering",
    "statistics": "statistics",
    "stats": "statistics",
    "regression": "regression",
    "clustering": "clustering",
    "data-viz": "data_visualization",
    "data visualization": "data_visualization",
    "data viz": "data_visualization",
    "matplotlib": "data_visualization",
    "tableau": "data_visualization",
    "power-bi": "data_visualization",
    "power bi": "data_visualization",
    "powerbi": "data_visualization",
    "pandas": "pandas",
    "numpy": "numpy",
    # Web — Frontend
    "react": "react",
    "reacts": "react",
    "reactjs": "react",
    "vue": "vue",
    "vue.js": "vue",
    "vuejs": "vue",
    "redux": "redux",
    "tailwind": "tailwind",
    "html/css": "html_css",
    "html css": "html_css",
    "html": "html_css",
    "css": "html_css",
    "jest": "jest",
    "graphql": "graphql",
    # Web — Backend
    "node.js": "nodejs",
    "nodejs": "nodejs",
    "node js": "nodejs",
    "flask": "flask",
    "spring boot": "spring_boot",
    "springboot": "spring_boot",
    "rest api": "rest_api",
    "rest": "rest_api",
    "restapi": "rest_api",
    "microservices": "microservices",
    # Databases
    "sql": "sql",
    "mysql": "mysql",
    "mysq": "mysql",
    "postgresql": "postgresql",
    "postgres": "postgresql",
    "mongodb": "mongodb",
    "redis": "redis",
    # DevOps / Cloud
    "docker": "docker",
    "kubernetes": "kubernetes",
    "kubernates": "kubernetes",
    "k8s": "kubernetes",
    "ci/cd": "ci_cd",
    "cicd": "ci_cd",
    "ci cd": "ci_cd",
    "aws": "aws",
    # Mobile
    "android": "android",
    "firebase": "firebase",
    # CS Fundamentals
    "algorithms": "algorithms",
    "algoritms": "algorithms",
    "data structure": "data_structures",
    "data structures": "data_structures",
    "competitive programming": "competitive_programming",
    # Design
    "ui/ux": "ui_ux",
    "ui ux": "ui_ux",
    "figma": "figma",
}


In [5]:

jd_data = [
    {"JD": "JD-1", "Company": "Kakao", "Role": "ML Engineer", "Required Skills": ["python", "machine learning", "deep learning", "tensorflow", "pytorch", "sql", "data visualization"]},
    {"JD": "JD-2", "Company": "Naver", "Role": "Backend Engineer", "Required Skills": ["java", "spring boot", "mysql", "postgresql", "microservices", "docker", "kubernetes"]},
    {"JD": "JD-3", "Company": "Line", "Role": "Frontend Engineer", "Required Skills": ["javascript", "react", "vue", "typescript", "rest api", "html/css"]},
]
resume_data = [
    {"ID": "01", "Candidate": "Arjun Sharma", "Raw Skills": ["pyhton", "machine learning", "sql", "pandas", "numpy", "deep learning"]},
    {"ID": "02", "Candidate": "Priya Nair", "Raw Skills": ["javascript", "react", "node.js", "mongodb", "rest api", "html/css"]},
    {"ID": "03", "Candidate": "Rahul Gupta", "Raw Skills": ["java", "spring boot", "mysql", "microservices", "docker", "kubernetes"]},
    {"ID": "04", "Candidate": "Sneha Patel", "Raw Skills": ["python", "tensorflow", "keras", "nlp", "bert", "data visualization"]},
    {"ID": "05", "Candidate": "Vikram Singh", "Raw Skills": ["c++", "algorithms", "data structures", "competitive programming", "python"]},
    {"ID": "06", "Candidate": "Ananya Krishnan", "Raw Skills": ["javascript", "vue.js", "python", "flask", "postgresql", "aws"]},
    {"ID": "07", "Candidate": "Karan Mehta", "Raw Skills": ["python", "sklearn", "xgboost", "feature engineering", "sql", "tableau"]},
    {"ID": "08", "Candidate": "Deepika Rao", "Raw Skills": ["java", "android", "kotlin", "firebase", "rest api", "ui/ux"]},
    {"ID": "09", "Candidate": "Aditya Kumar", "Raw Skills": ["reactjs", "typescript", "graphql", "redux", "tailwind", "nodejs"]},
    {"ID": "10", "Candidate": "Meera Iyer", "Raw Skills": ["python", "r", "statistics", "ml", "regression", "clustering"]},
]

In [6]:
#Normalize the skills in the resumes and job descriptions
for jd in jd_data:
    jd["Required Skills"] = [SKILL_ALIASES.get(skill, skill) for skill in jd["Required Skills"]]

for resume in resume_data:
    resume["Raw Skills"] = [SKILL_ALIASES.get(skill, skill) for skill in resume["Raw Skills"]]

In [7]:
#Create a vocabulary of all unique skills
vocabulary = set()
for jd in jd_data:
    vocabulary.update(jd["Required Skills"])
for resume in resume_data:
    vocabulary.update(resume["Raw Skills"])

In [10]:
#Calculate the TF-IDF vectors for the resumes
resume_vectors = []
for resume in resume_data:
    vector = {}
    for skill in vocabulary:
        # Term Frequency (TF)
        # Avoid division by zero if a resume has no skills (unlikely for this dataset)
        num_skills_in_resume = len(resume["Raw Skills"])
        tf = resume["Raw Skills"].count(skill) / num_skills_in_resume if num_skills_in_resume > 0 else 0

        # Document Frequency (DF)
        df = sum(1 for r in resume_data if skill in r["Raw Skills"])

        # Inverse Document Frequency (IDF)
        # Handle df = 0 to prevent ZeroDivisionError.
        # If a skill is not in any resume, its IDF contribution to scoring should be 0.
        if df == 0:
            idf = 0.0
        else:
            idf = math.log(len(resume_data) / df)

        vector[skill] = tf * idf
    resume_vectors.append(vector)

In [11]:
# Calculate the binary vectors for the job descriptions
jd_vectors = []
for jd in jd_data:
    vector = {}
    for skill in vocabulary:
        vector[skill] = 1 if skill in jd["Required Skills"] else 0
    jd_vectors.append(vector)

In [12]:
# Calculate the cosine similarity between the resumes and job descriptions
similarities = []
for i, resume_vector in enumerate(resume_vectors):
    for j, jd_vector in enumerate(jd_vectors):
        dot_product = sum(resume_vector[skill] * jd_vector[skill] for skill in vocabulary)
        magnitude_resume = math.sqrt(sum(resume_vector[skill] ** 2 for skill in vocabulary))
        magnitude_jd = math.sqrt(sum(jd_vector[skill] ** 2 for skill in vocabulary))
        similarity = dot_product / (magnitude_resume * magnitude_jd)
        similarities.append({
            "JD": jd_data[j]["JD"],
            "Candidate": resume_data[i]["Candidate"],
            "Similarity": similarity
        })

In [13]:
# Sort the similarities by JD and similarity
similarities.sort(key=lambda x: (x["JD"], -x["Similarity"]))

In [20]:
#Print the top 3 matches for each JD
for jd in [j['JD'] for j in jd_data]:
    print(f"{jd} — {jd_data[[j['JD'] for j in jd_data].index(jd)]['Company']} ({jd_data[[j['JD'] for j in jd_data].index(jd)]['Role']})")
    top_3 = [s for s in similarities if s["JD"] == jd][:3]
    top_3.sort(key=lambda x: (-x["Similarity"], x["Candidate"]))
    for i, similarity in enumerate(top_3):
        print(f"{i+1}. {similarity['Candidate']} (Similarity: {similarity['Similarity']:.2f})")
    print(" ")

JD-1 — Kakao (ML Engineer)
1. Arjun Sharma (Similarity: 0.47)
2. Karan Mehta (Similarity: 0.45)
3. Sneha Patel (Similarity: 0.34)
 
JD-2 — Naver (Backend Engineer)
1. Rahul Gupta (Similarity: 0.92)
2. Ananya Krishnan (Similarity: 0.18)
3. Deepika Rao (Similarity: 0.12)
 
JD-3 — Line (Frontend Engineer)
1. Priya Nair (Similarity: 0.64)
2. Ananya Krishnan (Similarity: 0.33)
3. Aditya Kumar (Similarity: 0.31)
 
